# 🏀 Multi-Camera Basketball Analysis
---

Welcome to the **Multi-Camera Basketball Analysis** notebook! Learn how to build a professional AI-powered basketball game analysis system with synchronized multi-camera coverage, real-time highlight detection via WebSocket, and automated multi-angle replay composition.

### 🎯 What You'll Build

A complete AI-powered multicam basketball analysis system that demonstrates:

- **👁️ SEE**: Connect to 3 synchronized cameras and preview live feeds
- **🧠 UNDERSTAND**: Index visual content across all cameras with AI analysis
- **🎬 ACT**: Detect events in real-time, receive WebSocket alerts, and create synchronized multi-angle videos

By the end, you'll have a working system that:
- Monitors 3 cameras simultaneously
- Detects specific events (slam dunks, three-pointers, fouls, fast breaks)
- Sends real-time alerts via WebSocket
- Creates synchronized multi-camera grid videos for replay review

---

## 🛠 Setup & Installation

Let's start by installing the VideoDB Python SDK and connecting to your account.

---

### 📦 Install VideoDB

VideoDB is available as a [Python package](https://pypi.org/project/videodb). Run the cell below to install it.

In [1]:
!pip install -q videodb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.5 MB/s eta 0:00:00


### 🔗 Connect to VideoDB

You'll need an API key to interact with VideoDB. Provide it securely when prompted below.

> 💡 **Tip:** Get your free API key from the [VideoDB Console](https://console.videodb.io). Get $20 in free API credits upon sign-up — no credit card required!

In [2]:
import videodb
import os
from getpass import getpass

api_key = getpass("Please enter your VideoDB API Key: ")
os.environ["VIDEO_DB_API_KEY"] = api_key

conn = videodb.connect()
coll = conn.get_collection()

print("✅ Connected to VideoDB successfully!")

Please enter your VideoDB API Key: ··········
✅ Connected to VideoDB successfully!


---

## 👁️ SEE - Connect Multi-Camera System

First, we'll connect to **3 synchronized cameras** monitoring a basketball arena from 3 camera angles. VideoDB handles RTSP stream ingestion seamlessly.

---

### 📹 Configure Cameras

In [3]:
# Multi-camera configuration
CAMERA_CONFIG = {
    "cam1": {"name": "Main Court Field", "url": "rtsp://samples.rts.videodb.io:8554/bb-cam1"},
    "cam2": {"name": "North Basket Area", "url": "rtsp://samples.rts.videodb.io:8554/bb-cam2"},
    "cam3": {"name": "South Basket Area", "url": "rtsp://samples.rts.videodb.io:8554/bb-cam3"}
}

print("📋 Camera Configuration:")
for cam_id, info in CAMERA_CONFIG.items():
    print(f"   {cam_id}: {info['name']}")

📋 Camera Configuration:
   cam1: Main Court Field
   cam2: North Basket Area
   cam3: South Basket Area


---
### Connect all streams

In [4]:
# Connect all cameras
print("🔌 Connecting to all cameras...\n")

streams = {}
for cam_id, cam_info in CAMERA_CONFIG.items():
    stream = coll.connect_rtstream(
        name=f"Basketball_{cam_id}",
        url=cam_info["url"],
        store=True,  # enables recording storage
    )
    streams[cam_id] = {"stream": stream, "info": cam_info}
    print(f"✅ {cam_id}: {stream.id}")

print(f"\n✅ All {len(streams)} cameras connected!")

🔌 Connecting to all cameras...

✅ cam1: rts-019e9c87-5394-7261-8ef7-90093e9129a9
✅ cam2: rts-019e9c87-5502-7052-b042-596f3ef2b43c
✅ cam3: rts-019e9c87-5611-7c33-aa81-52756c077b7c

✅ All 3 cameras connected!


#### To reconnect to existing stream:

In [5]:
# EXISTING_STREAM_IDS = {
#     "cam1": "rts-xxxxx-xxxxx-xxxxx",  # Replace with your cam1 stream ID
#     "cam2": "rts-xxxxx-xxxxx-xxxxx",  # Replace with your cam2 stream ID
#     "cam3": "rts-xxxxx-xxxxx-xxxxx",  # Replace with your cam3 stream ID
#     "cam4": "rts-xxxxx-xxxxx-xxxxx",  # Replace with your cam4 stream ID
# }

# print("🔌 Reconnecting to existing cameras...\n")

# streams = {}
# for cam_id, rtstream_id in EXISTING_STREAM_IDS.items():
#     stream = coll.get_rtstream(rtstream_id)
#     streams[cam_id] = {"stream": stream, "info": CAMERA_CONFIG[cam_id]}
#     print(f"✅ {cam_id}: {stream.id} ({CAMERA_CONFIG[cam_id]['name']})")

# print(f"\n✅ All {len(streams)} cameras reconnected!")

---

### 📺 Preview Live Feeds

Let's preview the last 2 minutes from each camera to verify connectivity:
> Wait for atleast 2 minuts before executing the following cell.

In [19]:
from IPython.display import HTML
import time

# Get timestamps for last 2 minutes
now = int(time.time())
ten_seconds_ago = now - 600  # 10 min lookback for more buffer

print("📹 Generating preview streams for all cameras...\n")

# Generate streams and collect stream URLs
stream_urls = {}
video_titles = []
for cam_id, cam_data in streams.items():
    stream = cam_data["stream"]

    # Must run both (critical for RTStream preview)
    player_url = stream.generate_stream(ten_seconds_ago, now)

    stream_url = stream.stream_url
    stream_urls[cam_id] = stream_url
    video_titles.append(cam_data['info']['name'])
    print(f"✅ {cam_data['info']['name']}")
    print(f"  {player_url}\n")

print("\n📺 Displaying all 3 camera feeds:\n")

# Create HTML with 3 videos in a row
html_content = """
<div style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 10px; max-width: 900px;">
"""

for i, (cam_id, url) in enumerate(stream_urls.items()):
    html_content += f'''
        <div style="text-align: center;">
            <h4>{video_titles[i]}</h4>
            <iframe src="{url}" width="100%" height="200" frameborder="0" allowfullscreen></iframe>
        </div>
    '''

html_content += '</div>'

display(HTML(html_content))

📹 Generating preview streams for all cameras...

✅ Main Court Field
  https://player.videodb.io/watch?v=wDMzFKFKSJQ

✅ North Basket Area
  https://player.videodb.io/watch?v=Pl7WIKq0dfQ

✅ South Basket Area
  https://player.videodb.io/watch?v=koQwtUDJnZo


📺 Displaying all 3 camera feeds:



---

## 🧠 UNDERSTAND - AI-Powered Visual Analysis

Now we'll set up AI-powered visual indexing across all 3 cameras. The AI will analyze frames every 10 seconds to detect basketball actions and plays.

---

### 🔍 Configure Visual Indexing

In [20]:
# Scene indexing configuration
SCENE_INDEX_CONFIG = {
    "batch_config": {
        "type": "time",
        "value": 10,  # Analyze every 10 seconds
        "frame_count": 1
    },
    "prompt": """Analyze this basketball game footage. Describe:
    1. Slam dunks, three-point shots, and scoring plays
    2. Fast breaks and quick transition plays
    3. Fouls, physical contact, or player collisions
    Be specific about player positions and court location."""
}

print("📋 Visual Index Configuration:")
print(f"   Analyzing every {SCENE_INDEX_CONFIG['batch_config']['value']} seconds")
print(f"   AI Focus: Basketball actions (dunks, shots, fast breaks, fouls)")

📋 Visual Index Configuration:
   Analyzing every 10 seconds
   AI Focus: Basketball actions (dunks, shots, fast breaks, fouls)


In [21]:
# Index visuals for all cameras
print("🔧 Creating visual indexes...\n")

scene_indexes = {}
for cam_id, cam_data in streams.items():
    stream = cam_data["stream"]
    scene_index = stream.index_visuals(
        batch_config=SCENE_INDEX_CONFIG["batch_config"],
        prompt=SCENE_INDEX_CONFIG["prompt"],
        name=f"Basketball_{cam_id}_Index"
    )
    scene_indexes[cam_id] = {
        "index": scene_index,
        "index_id": scene_index.rtstream_index_id
    }
    print(f"✅ {cam_id}: {scene_index.rtstream_index_id}")

print(f"\n✅ Visual indexing active on all {len(scene_indexes)} cameras!")

🔧 Creating visual indexes...

✅ cam1: 20921bf87ba0ad49
✅ cam2: 2c5f040e24021984
✅ cam3: 4da630d6b38d8d19

✅ Visual indexing active on all 3 cameras!


#### If you've already created scene indexes, you can reconnect to them instead of creating new ones.

In [9]:
# EXISTING_INDEX_IDS = {
#     "cam1": "idx-xxxxx-xxxxx-xxxxx",  # Replace with your cam1 index ID
#     "cam2": "idx-xxxxx-xxxxx-xxxxx",  # Replace with your cam2 index ID
#     "cam3": "idx-xxxxx-xxxxx-xxxxx",  # Replace with your cam3 index ID
#     "cam4": "idx-xxxxx-xxxxx-xxxxx",  # Replace with your cam4 index ID
# }
#
# print("🔧 Reconnecting to existing scene indexes...\n")
#
# scene_indexes = {}
# for cam_id, index_id in EXISTING_INDEX_IDS.items():
#     stream = streams[cam_id]["stream"]
#     scene_index = stream.get_scene_index(index_id)
#     scene_indexes[cam_id] = {
#         "index": scene_index,
#         "index_id": scene_index.rtstream_index_id
#     }
#     print(f"✅ {cam_id}: {scene_index.rtstream_index_id}")
#
# print(f"\n✅ Reconnected to all {len(scene_indexes)} scene indexes!")

---

### 👀 Review Indexed Scenes

Let's see what the AI has detected so far. We'll look at scenes from two different camera angles:

In [22]:
def display_scenes(cam_id, num_scenes=3, max_chars=200):
    """Display indexed scenes from a specific camera (compact format)"""
    cam_name = streams[cam_id]["info"]["name"]
    scene_index = scene_indexes[cam_id]["index"]

    print(f"📹 {cam_name}:")

    scenes_data = scene_index.get_scenes(page_size=num_scenes)

    if scenes_data and scenes_data.get("scenes"):
        for i, scene in enumerate(scenes_data.get("scenes"), 1):
            start = scene.get("start", 0)
            description = scene.get("description", "No description")

            # Clean up description: remove extra whitespace and newlines
            clean_desc = " ".join(description.split())

            # Truncate if too long
            if len(clean_desc) > max_chars:
                clean_desc = clean_desc[:max_chars].rsplit(' ', 1)[0] + "..."

            print(f"  └─ Scene {i} [{int(start)}s]: {clean_desc}")
        print()  # Single newline after all scenes
    else:
        print("  └─ No scenes indexed yet. Wait a few moments.\n")

---
#### 👀 Cam 1 : Main Court Field

In [27]:
# Camera 1 - Main Court Field
display_scenes("cam1")

# Camera 2 - North Basket Area
display_scenes("cam2")

# Camera 3 - South Basket Area
display_scenes("cam3")

📹 Main Court Field:
  └─ Scene 1 [1780742661s]: Based on the provided still image, here's an analysis of the basketball footage: **General Observation:** The image captures a dynamic moment during a basketball game, likely a transition play, with...
  └─ Scene 2 [1780742649s]: Here's an analysis of the basketball game footage based on the provided image: **1. Slam dunks, three-point shots, and scoring plays:** * **Scoring Play:** The primary action is a **scoring...
  └─ Scene 3 [1780742637s]: This is a still image from a basketball game, which means I cannot observe dynamic actions like slam dunks, three-point shots, fast breaks, or fouls in progress. The image captures a specific moment...

📹 North Basket Area:
  └─ Scene 1 [1780742665s]: Based on the provided image, here's an analysis of the basketball game footage: **1. Slam dunks, three-point shots, and scoring plays:** * **Slam dunks:** There are no slam dunks visible in this...
  └─ Scene 2 [1780742651s]: Based on the provided s

💡 **Multi-angle coverage is active!** All 3 cameras are continuously analyzing and understanding what's happening on the basketball court.

---

---

## 🎬 ACT - Event Detection & Multi-Angle Composition

Now comes the exciting part! We'll:
1. Create event detection rules
2. Monitor for alerts in real-time via WebSocket
3. Create synchronized multi-camera evidence videos

---

### 🚨 Define Events to Detect

In [28]:
# Define 4 basketball events
EVENTS_CONFIG = [
    {"label": "slam_dunk", "prompt": "Detect when a player performs a slam dunk"},
    {"label": "three_pointer", "prompt": "Detect when a player scores a three-point shot"},
    {"label": "foul", "prompt": "Detect when a foul or physical contact occurs"},
    {"label": "fast_break", "prompt": "Detect a fast break or quick transition play"}
]

print("🎯 Creating event detection rules...\n")

events = {}
for cfg in EVENTS_CONFIG:
    event_id = conn.create_event(
        event_prompt=cfg["prompt"],
        label=cfg["label"]
    )
    events[cfg["label"]] = {"event_id": event_id}
    print(f"Event: {cfg['label']}")
    print(f"    ID: {event_id}")

print(f"\n✅ {len(events)} events ready for detection!")

🎯 Creating event detection rules...

Event: slam_dunk
    ID: 5a5fbde68f58be45
Event: three_pointer
    ID: 425c3cc8dfe1ac5e
Event: foul
    ID: f0af7f4893bdb2e3
Event: fast_break
    ID: ff15647fd3ae8ed9

✅ 4 events ready for detection!


---

### 🔌 Connect WebSocket for Real-Time Alerts

WebSockets let us receive alerts instantly as they happen:

In [29]:
import asyncio

# Connect to WebSocket
ws_wrapper = conn.connect_websocket()
ws = await ws_wrapper.connect()

print(f"✅ WebSocket connected!")
print(f"   Connection ID: {ws.connection_id}")

INFO:videodb.websocket_client:WebSocket connected with ID: gQHRgk47keO4KAIZ_A==


✅ WebSocket connected!
   Connection ID: gQHRgk47keO4KAIZ_A==


Create alerts for every camera × event combination

In [30]:
# Create alerts for all cameras × all events
print("🔔 Creating alerts...\n")

alerts = {}
for cam_id, idx_data in scene_indexes.items():
    alerts[cam_id] = {}
    print(f"--- Alerts for Camera: {cam_id} ---\n")
    for label, evt in events.items():
        alert_id = idx_data["index"].create_alert(
            evt["event_id"],
            callback_url="",  # Empty = WebSocket only
            ws_connection_id=ws.connection_id
        )
        alerts[cam_id][label] = alert_id
        print(f"Alert: {label}")
        print(f"    ID: {alert_id}")
    print(f"\n")

total_alerts = len(alerts) * len(events)
print(f"\n✅ {total_alerts} alerts active (3 cameras × 4 events)")



🔔 Creating alerts...

--- Alerts for Camera: cam1 ---

Alert: slam_dunk
    ID: a1b6dec997507215
Alert: three_pointer
    ID: b3d3deeaf68ba995
Alert: foul
    ID: cb837bec9fb18232
Alert: fast_break
    ID: e5a6f426725fdd2b


--- Alerts for Camera: cam2 ---

Alert: slam_dunk
    ID: 7472f745885f5027
Alert: three_pointer
    ID: b55cea60786bb38c
Alert: foul
    ID: a29d18e57a03375a
Alert: fast_break
    ID: 0486697e84c27321


--- Alerts for Camera: cam3 ---

Alert: slam_dunk
    ID: 6cbf1c1e8f99a254
Alert: three_pointer
    ID: fdde5ee41671c95e
Alert: foul
    ID: 8ad5814858897cf3
Alert: fast_break
    ID: c09e966d9c20f9fa



✅ 12 alerts active (3 cameras × 4 events)


---

### 👂 Listen for Alerts

Let's listen for 30 seconds. When events are detected, we'll receive instant notifications:

In [31]:
# Create mapping from rtstream_id to cam info for easy lookup
rtstream_to_cam = {
    streams[cam_id]["stream"].id: {"cam_id": cam_id, "cam_name": CAMERA_CONFIG[cam_id]["name"]}
    for cam_id in streams.keys()
}

# Store alerts for later analysis
received_alerts = []

async def listen_for_alerts():
    timeout = 30  # Adjustable
    print(f"⏱️ Listening for alerts ({timeout} seconds)...")
    print("💡 The basketball analysis system will trigger alerts when events are detected\n")

    try:
        async with asyncio.timeout(timeout):
            async for msg in ws.receive():
                if msg.get("channel") == "alert":
                    received_alerts.append(msg)
                    alert_num = len(received_alerts)
                    data = msg.get("data", {})

                    # Get camera info
                    rtstream_id = msg.get("rtstream_id", "unknown")
                    cam_info = rtstream_to_cam.get(rtstream_id, {"cam_id": "unknown", "cam_name": "Unknown"})

                    print(f"\n🚨 ALERT #{alert_num} RECEIVED!")
                    print(f"   Camera: {cam_info['cam_id']} - {cam_info['cam_name']}")
                    print(f"   Event: {data.get('label', 'N/A')}")
                    print(f"   Time: {msg.get('timestamp', 'N/A')}")
                    print(f"   Confidence: {data.get('confidence', 'N/A')}")
                    print(f"   📹 Clip URL: {data.get('stream_url', 'N/A')}")
                    print(f"   💡 Explanation: {data.get('explanation', 'N/A')[:150]}...")  # Truncate explanation

    except asyncio.TimeoutError:
        print(f"\n✅ Listening complete! {len(received_alerts)} alert(s) received")

# Start listening
await listen_for_alerts()

⏱️ Listening for alerts (30 seconds)...
💡 The basketball analysis system will trigger alerts when events are detected


🚨 ALERT #1 RECEIVED!
   Camera: cam2 - North Basket Area
   Event: fast_break
   Time: 2026-06-06T10:45:34.985861+00:00
   Confidence: 0.95
   📹 Clip URL: https://rt.stream.videodb.io/manifests/rts-019e9c87-5502-7052-b042-596f3ef2b43c/1780742705000000-1780742717000000.m3u8
   💡 Explanation: The scene analysis explicitly concludes that the image strongly suggests a fast break or quick transition play is in progress, citing players sprintin...

🚨 ALERT #2 RECEIVED!
   Camera: cam1 - Main Court Field
   Event: foul
   Time: 2026-06-06T10:45:48.482606+00:00
   Confidence: 0.9
   📹 Clip URL: https://rt.stream.videodb.io/manifests/rts-019e9c87-5394-7261-8ef7-90093e9129a9/1780742715000000-1780742727000000.m3u8
   💡 Explanation: The scene analysis explicitly states 'Physical Contact (Legal)' between players under the basket, indicating that physical contact is occurring, even

---

### 📊 Analyze Alert Metrics

In [32]:
# Count alerts by event type
by_type = {}
for alert in received_alerts:
    label = alert.get("data", {}).get("label", "unknown")
    by_type[label] = by_type.get(label, 0) + 1

print("📊 Alerts by Event Type:")
print("-" * 50)
for label, count in sorted(by_type.items(), key=lambda x: x[1], reverse=True):
    # Format label: slam_dunk -> Slam Dunk
    formatted_label = label.replace("_", " ").title()
    print(f"   {formatted_label:.<35} {count}")

# Count alerts by camera (using rtstream_to_cam mapping)
by_cam = {}
for alert in received_alerts:
    rtstream_id = alert.get("rtstream_id", "unknown")
    cam_info = rtstream_to_cam.get(rtstream_id, {"cam_id": "unknown", "cam_name": "Unknown"})
    cam_key = f"{cam_info['cam_id']} - {cam_info['cam_name']}"
    by_cam[cam_key] = by_cam.get(cam_key, 0) + 1

print("\n📹 Alerts by Camera:")
print("-" * 50)
for cam, count in sorted(by_cam.items(), key=lambda x: x[1], reverse=True):
    print(f"   {cam:.<35} {count}")

print(f"\n✅ Total: {len(received_alerts)} alerts received")

📊 Alerts by Event Type:
--------------------------------------------------
   Fast Break......................... 1
   Foul............................... 1

📹 Alerts by Camera:
--------------------------------------------------
   cam2 - North Basket Area........... 1
   cam1 - Main Court Field............ 1

✅ Total: 2 alerts received


---

### 🔍 Select Alert to Investigate

Let's investigate a "fast break" alert and create a multi-angle evidence video:

In [33]:
# Auto-select first fast break alert
selected = next(
    (a for a in received_alerts if a["data"]["label"] == "fast_break"),
    None
)

if selected:
    data = selected["data"]
    print(f"🚨 Selected Alert: {data['label']}")
    print(f"   Confidence: {data['confidence']}")
    print(f"   Time: {selected.get('timestamp')}")
else:
    print("⚠️ No fast break alerts received. Try running for longer or select different event.")

🚨 Selected Alert: fast_break
   Confidence: 0.95
   Time: 2026-06-06T10:45:34.985861+00:00


---

### 🎥 Generate Synchronized Multi-Camera Clips

Now we'll extract the same moment from all 3 cameras:

In [34]:
# Extract timestamp from alert
data = selected["data"]
start_time = data.get("start")  # Unix timestamp
end_time = data.get("end")

# Add 10 second padding for context
PADDING = 10
clip_start = int(start_time - PADDING)
clip_end = int(end_time + PADDING)
clip_duration = clip_end - clip_start

print(f"⏱️ Alert Time Window: {start_time} → {end_time}")
print(f"📹 With padding: {clip_start} → {clip_end}\n")

# Generate synchronized streams for all cameras
stream_urls = {}
player_urls_map = {}
for cam_id, cam_data in streams.items():
    stream = cam_data["stream"]

    # Must run both (critical for RTStream)
    player_url = stream.generate_stream(clip_start, clip_end)
    stream_url = stream.stream_url

    stream_urls[cam_id] = stream_url
    player_urls_map[cam_id] = player_url

print(f"✅ 3 synchronized clips ready\n")

print("Player URLs for each camera:")
for cam_id, p_url in player_urls_map.items():
    cam_name = CAMERA_CONFIG[cam_id]["name"]
    print(f"{cam_id} : {cam_name}")
    print(f"    {p_url}")

⏱️ Alert Time Window: 1780742705.9603114 → 1780742716.970289
📹 With padding: 1780742695 → 1780742726

✅ 3 synchronized clips ready

Player URLs for each camera:
cam1 : Main Court Field
    https://player.videodb.io/watch?v=PE8glUo4IKs
cam2 : North Basket Area
    https://player.videodb.io/watch?v=UCKKaMOIk-o
cam3 : South Basket Area
    https://player.videodb.io/watch?v=tkzjRNkjUOQ


---

### 💾 Download Clips for Composition

Since the Timeline Editor requires VideoDB media IDs, we'll download these clips using ffmpeg.

In [35]:
import subprocess
import os

# Download all 3 clips using ffmpeg
print("📥 Downloading clips with ffmpeg...\n")

downloads = {}
for cam_id, stream_url in stream_urls.items():
    output_file = f"{cam_id}_clip.mp4"

    # Use ffmpeg to download HLS stream
    cmd = [
        'ffmpeg',
        '-y',  # Overwrite output file
        '-i', stream_url,  # Input HLS URL
        '-c', 'copy',  # Copy codec (no re-encoding)
        '-loglevel', 'error',  # Only show errors
        output_file
    ]

    print(f"   {cam_id}: Downloading...")
    try:
        subprocess.run(cmd, check=True, capture_output=True)
        downloads[cam_id] = {"name": output_file}
        print(f"   ✅ {cam_id}: Saved as {output_file}\n")
    except subprocess.CalledProcessError as e:
        print(f"   ❌ {cam_id}: Failed - {e.stderr.decode()}")
        downloads[cam_id] = {"name": None, "error": str(e)}

print(f"✅ Downloaded {len([d for d in downloads.values() if d.get('name')])} clips!")

📥 Downloading clips with ffmpeg...

   cam1: Downloading...
   ✅ cam1: Saved as cam1_clip.mp4

   cam2: Downloading...
   ✅ cam2: Saved as cam2_clip.mp4

   cam3: Downloading...
   ✅ cam3: Saved as cam3_clip.mp4

✅ Downloaded 3 clips!


#### 💾 Uploading back to VideoDB

In [36]:
# Upload clips back to VideoDB
print("📤 Uploading clips to VideoDB...\n")

videos = {}
for cam_id, dl_info in downloads.items():
    video = coll.upload(file_path=dl_info["name"])
    videos[cam_id] = {"video": video, "video_id": video.id}
    print(f"✅ {cam_id}: {video.id}\n")

print("✅ All clips uploaded and ready for composition!")

📤 Uploading clips to VideoDB...

✅ cam1: m-z-019e9c8a-76fd-73d2-96ba-0ed7c1da8fbd

✅ cam2: m-z-019e9c8a-c5f9-7273-8857-2d2a51b8da1f

✅ cam3: m-z-019e9c8a-f05b-77b1-ba06-6f9a0db510e6

✅ All clips uploaded and ready for composition!


---
### 🎬 Create 3-Camera Replay Grid

Now composing all 3 camera angles into a synchronized multi-camera replay:
- **Cam 1 (Main Court Field)**: Top-left
- **Cam 2 (North Basket Area)**: Top-right
- **Cam 3 (South Basket Area)**: Bottom-center

In [37]:
from videodb.editor import (
    Timeline, Track, Clip, VideoAsset, Position, Offset, Fit,
    TextAsset, Font, Border, Shadow, Background, TextAlignment
)

# Get minimum duration
print("📏 Checking video durations...\n")
video_durations = {}
for cam_id, vid_info in videos.items():
    video = vid_info["video"]
    video_durations[cam_id] = video.length

final_duration = min(video_durations.values())

# Create timeline with medium gray background
timeline = Timeline(conn)
timeline.resolution = "1280x720"
timeline.background = "#404040"

# Camera positions: top-left, top-right, bottom-center
cam_configs = [
    {"cam_id": "cam1", "position": Position.top_left, "offset": Offset(x=0.03, y=0.025)},
    {"cam_id": "cam2", "position": Position.top_right, "offset": Offset(x=-0.03, y=0.025)},
    {"cam_id": "cam3", "position": Position.bottom, "offset": Offset(x=0, y=-0.025)},
]

print("🎬 Building multi-camera grid...\n")

# Add video tracks
for config in cam_configs:
    cam_id = config["cam_id"]
    clip = Clip(
        asset=VideoAsset(id=videos[cam_id]["video_id"]),
        duration=final_duration,
        fit=Fit.crop,
        position=config["position"],
        offset=config["offset"],
        scale=0.45,
    )
    track = Track()
    track.add_clip(0, clip)
    timeline.add_track(track)

# Label positions matching each camera
label_configs = [
    {"cam_id": "cam1", "offset": Offset(x=-0.355, y=-0.45)},
    {"cam_id": "cam2", "offset": Offset(x=0.135, y=-0.45)},
    {"cam_id": "cam3", "offset": Offset(x=0, y=0.05)},
]

# Add camera labels
for config in label_configs:
    cam_id = config["cam_id"]
    cam_name = CAMERA_CONFIG[cam_id]["name"]

    label_text = TextAsset(
        text=f"{cam_id.upper()}: {cam_name}",
        font=Font(family="Clear Sans", size=24, color="#FFFFFF"),
        background=Background(
            color="#000000",
            height=50,
            width=300,
            text_alignment=TextAlignment.center,
        ),
    )

    label_clip = Clip(
        asset=label_text,
        duration=final_duration,
        offset=config["offset"],
    )

    track = Track()
    track.add_clip(0, label_clip)
    timeline.add_track(track)

print(f"✅ Multi-camera replay ready!\n")
print(f"   Resolution: 1280x720")
print(f"   Duration: {final_duration:.2f}s")
print(f"   Layout: Top row (Main Court + North Basket) + Bottom center (South Basket)")

📏 Checking video durations...

🎬 Building multi-camera grid...

✅ Multi-camera replay ready!

   Resolution: 1280x720
   Duration: 16.00s
   Layout: Top row (Main Court + North Basket) + Bottom center (South Basket)


In [38]:
# Generate final multi-camera view
from videodb import play_stream

print("🎬 Generating final video...\n")

stream = timeline.generate_stream()

print("✅ Multi-camera synchronized view ready!")
print(f"Stream: {stream}\n")

play_stream(stream)

🎬 Generating final video...

✅ Multi-camera synchronized view ready!
Stream: https://play.videodb.io/v1/63f5ce20-616f-40cf-8101-865831bca2aa.m3u8



<div style="background-color: #d4edda; color: #155724; padding: 12px; border-left: 5px solid #28a745; border-radius: 4px;">
    <strong>🎉 Success!</strong> You've created a professional synchronized multi-camera basketball replay — showing the same highlight from 3 different angles!
</div>

---

## 🧹 Cleanup

When you're done, let's disconnect all resources:

In [39]:
print("🧹 Cleaning up resources...\n")

# Close WebSocket
await ws_wrapper.close()
print("✅ WebSocket closed")

# Disable all alerts
for cam_id, cam_alerts in alerts.items():
    for label, alert_id in cam_alerts.items():
        try:
            scene_indexes[cam_id]["index"].disable_alert(alert_id)
        except Exception as e:
            pass

print("✅ All alerts disabled")

# Stop all streams
for cam_id, cam_data in streams.items():
    try:
        cam_data["stream"].stop()
    except Exception as e:
        pass

print("✅ All streams stopped")
print("\n🎉 Cleanup complete!")

🧹 Cleaning up resources...

✅ WebSocket closed
✅ All alerts disabled
✅ All streams stopped

🎉 Cleanup complete!


---

# 🏁 Conclusion: Multi-Camera Basketball Analysis

Congratulations! You've built a complete AI-powered multi-camera basketball analysis system.

### 🎯 What You Accomplished

**👁️ SEE:**
- ✅ Connected 3 synchronized RTSP camera streams
- ✅ Previewed live feeds from multiple angles

**🧠 UNDERSTAND:**
- ✅ Indexed visual content across all cameras with AI
- ✅ Reviewed scene descriptions from different angles

**🎬 ACT:**
- ✅ Created 4 event detection rules
- ✅ Monitored real-time alerts via WebSocket
- ✅ Analyzed alert metrics by type and camera
- ✅ Generated synchronized multi-camera clips
- ✅ Composed a professional multi-camera grid video

---

### 🚀 What's Next?

Ready to build more advanced sports analysis systems?

- 📖 **[VideoDB Documentation](https://docs.videodb.io)**: Complete API reference
- 🍳 **[VideoDB Cookbook](https://github.com/video-db/videodb-cookbook)**: More multicam examples
- 💬 **[Discord Community](https://discord.com/invite/py9P639jGz)**: Get help and share projects

---

**Build intelligent sports analysis systems with VideoDB — real-time AI for every game.** 🏀

